In [1]:
# AI JOB ANALYST — BASIC VERSION
# Rule-based CV → Job Description matching and evidence analysis.
# GitHub-ready notebook: works with a local CV PDF path.
# Install dependencies from requirements.txt before running.


In [ ]:
# 1. CV INPUT
# Put your CV PDF in the same folder as this notebook and set its filename below.
from pathlib import Path

filename = "YOUR_CV.pdf"

if not Path(filename).exists():
    raise FileNotFoundError(
        f"CV file not found: {filename}. "
        "Place the PDF beside this notebook or update 'filename'."
    )

print("Using CV:", filename)


In [ ]:
# 2. READ CV PDF
from pypdf import PdfReader
import re
import numpy as np

reader = PdfReader(filename)

cv_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        cv_text += text + "\n"

print("Pages:", len(reader.pages))
print("Characters:", len(cv_text))


In [4]:
# 3. Clean CV and extract sections
def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


clean_cv = clean_text(cv_text)


def extract_cv_sections(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    section_names = [
        "Professional Summary",
        "Work Experience",
        "Education",
        "Projects",
        "Technical Skills",
        "Leadership & Achievements"
    ]

    pattern = r"\b(" + "|".join(
        re.escape(x) for x in section_names
    ) + r")\b"

    matches = list(re.finditer(pattern, text, flags=re.I))
    sections = {}

    for i, match in enumerate(matches):
        section_name = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[section_name] = text[start:end].strip()

    return sections


cv_sections = extract_cv_sections(cv_text)

cv_profile = {
    "summary": cv_sections.get("Professional Summary", ""),
    "experience": cv_sections.get("Work Experience", ""),
    "education": cv_sections.get("Education", ""),
    "projects": cv_sections.get("Projects", ""),
    "technical_skills": cv_sections.get("Technical Skills", ""),
    "leadership": cv_sections.get("Leadership & Achievements", "")
}

print("CV sections:", list(cv_profile.keys()))




CV sections: ['summary', 'experience', 'education', 'projects', 'technical_skills', 'leadership']


In [5]:
# 4. Skill aliases
skill_aliases = {
    "Python": ["python", "python programming"],
    "SQL": ["sql", "structured query language"],
    "Excel": ["excel", "advanced excel"],
    "Power BI": ["power bi", "powerbi"],
    "Tableau": ["tableau"],
    "R": ["r programming", "r language", "tidyverse"],
    "Pandas": ["pandas"],
    "NumPy": ["numpy"],
    "Machine Learning": ["machine learning"],
    "Statistics": ["statistics", "statistical"],
    "Statistical Modeling": ["statistical modeling"],
    "Statistical Testing": ["statistical testing"],
    "Hypothesis Testing": [
        "hypothesis testing", "hypothesis-driven", "hypothesis",
        "chi-square", "chi square", "mann-whitney", "mann–whitney",
        "fisher's exact", "fisher exact", "wilcoxon", "bowker",
        "proportion tests"
    ],
    "Exploratory Data Analysis": [
        "exploratory data analysis", "exploratory analysis", "exploratory"
    ],
    "Data Visualization": [
        "data visualization", "visualization", "dashboard"
    ],
    "Regression": ["regression", "logistic regression"],
    "Classification": ["classification"],
    "Random Forest": ["random forest"],
    "XGBoost": ["xgboost"],
    "SHAP": ["shap"],
    "Feature Engineering": ["feature engineering"],
    "Model Evaluation": ["model evaluation"],
    "Forecasting": ["forecasting", "forecast"],
    "Time Series Forecasting": ["time series forecasting"],
    "NLP": ["natural language processing", "nlp"],
    "Text Processing": ["text processing"],
    "Information Extraction": ["information extraction"],
    "Regex": ["regex", "regular expression"],
    "Data Analysis": ["data analysis", "data analytics"],
    "Data Processing": ["data processing"],
    "Data Cleaning": ["data cleaning", "cleaned", "cleansed"],
    "Data Validation": ["data validation", "validated", "validation"],
    "Large Datasets": ["large dataset", "large datasets"],
    "Decision Support": [
        "decision support", "decision making", "decision-making"
    ],
    "Analytical Reporting": [
        "analytical report", "analytical reports", "analytical outputs"
    ],
    "Sensitivity Analysis": [
        "sensitivity analysis", "sensitivity analyses"
    ]
}


def extract_skills_with_aliases(text, aliases):
    text_lower = text.lower()
    found = []

    for skill, terms in aliases.items():
        if any(term.lower() in text_lower for term in terms):
            found.append(skill)

    return sorted(set(found))


# Scan the ENTIRE CV so skills demonstrated in projects and work experience
# are also recognized.
cv_skills = extract_skills_with_aliases(
    cv_text,
    skill_aliases
)

# Power BI dashboard is also evidence of Data Visualization.
if "Power BI" in cv_skills:
    cv_skills.append("Data Visualization")

cv_skills = sorted(set(cv_skills))

print("\nDetected CV skills:")
for skill in cv_skills:
    print("✓", skill)





Detected CV skills:
✓ Analytical Reporting
✓ Classification
✓ Data Analysis
✓ Data Cleaning
✓ Data Validation
✓ Data Visualization
✓ Decision Support
✓ Excel
✓ Exploratory Data Analysis
✓ Forecasting
✓ Hypothesis Testing
✓ Information Extraction
✓ Large Datasets
✓ Machine Learning
✓ Model Evaluation
✓ NLP
✓ NumPy
✓ Pandas
✓ Power BI
✓ Python
✓ R
✓ Random Forest
✓ Regex
✓ Regression
✓ SHAP
✓ SQL
✓ Sensitivity Analysis
✓ Statistical Modeling
✓ Statistical Testing
✓ Statistics
✓ Text Processing
✓ Time Series Forecasting
✓ XGBoost


In [6]:
# 5. Calculate total work experience
experience_text = cv_profile["experience"]

date_pattern = r"""
(
    Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec
)
\s+(\d{4})
\s*[-–]\s*
(
    Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec
)
\s+(\d{4})
"""

matches = re.findall(
    date_pattern,
    experience_text,
    flags=re.IGNORECASE | re.VERBOSE
)

month_numbers = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4,
    "may": 5, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "oct": 10, "nov": 11, "dec": 12
}


def calculate_months(start_month, start_year, end_month, end_year):
    start = int(start_year) * 12 + month_numbers[start_month.lower()]
    end = int(end_year) * 12 + month_numbers[end_month.lower()]
    return end - start + 1


experience_periods = []

for match in matches:
    start_month, start_year, end_month, end_year = match

    months = calculate_months(
        start_month,
        start_year,
        end_month,
        end_year
    )

    experience_periods.append({
        "start": f"{start_month} {start_year}",
        "end": f"{end_month} {end_year}",
        "months": months,
        "years": round(months / 12, 2)
    })


cv_experience_profile = {
    "total_months": sum(x["months"] for x in experience_periods),
    "total_years": round(
        sum(x["months"] for x in experience_periods) / 12,
        2
    ),
    "periods": experience_periods
}

print("\nExperience:", cv_experience_profile)





Experience: {'total_months': 18, 'total_years': 1.5, 'periods': [{'start': 'Aug 2025', 'end': 'Jan 2026', 'months': 6, 'years': 0.5}, {'start': 'Jan 2024', 'end': 'Dec 2024', 'months': 12, 'years': 1.0}]}


In [7]:
# 6. Work-experience evidence
def extract_work_bullets(text):
    parts = re.split(r"\s*•\s*", text)
    bullets = []

    for part in parts:
        part = part.strip()

        if part:
            bullets.append({
                "title": "Work Experience",
                "years": cv_experience_profile["total_years"],
                "text": part
            })

    return bullets


experience_bullets = extract_work_bullets(
    cv_profile["experience"]
)

if not experience_bullets:
    experience_bullets = [{
        "title": "Work Experience",
        "years": cv_experience_profile["total_years"],
        "text": cv_profile["experience"]
    }]


for bullet in experience_bullets:
    bullet["skills"] = extract_skills_with_aliases(
        bullet["text"],
        skill_aliases
    )




In [8]:
# 7. Project evidence
def extract_projects_from_cv(text):
    text = text.replace("\xa0", " ")
    text = text.replace("sup- porting", "supporting")
    text = text.replace("customer- risk", "customer-risk")
    text = re.sub(r"\s+", " ", text).strip()

    projects = []

    # Limit extraction to the Projects section.
    project_match = re.search(
        r"\bProjects\b(.*?)(?=\bTechnical Skills\b|\bLeadership\b|$)",
        text,
        flags=re.IGNORECASE
    )

    project_text = (
        project_match.group(1).strip()
        if project_match
        else text
    )

    project_titles = [
        "Customer Churn Prediction & Statistical Analysis",
        "Analysis of Alcohol Use Disorder Data",
        "Intelligent CV Job Description Matching System"
    ]

    found_titles = []

    for title in project_titles:
        match = re.search(
            re.escape(title),
            project_text,
            flags=re.IGNORECASE
        )
        if match:
            found_titles.append((match.start(), title, match.end()))

    found_titles.sort()

    for i, (start_pos, title, title_end) in enumerate(found_titles):
        if i + 1 < len(found_titles):
            end_pos = found_titles[i + 1][0]
        else:
            end_pos = len(project_text)

        body = project_text[title_end:end_pos].strip()

        # Remove date prefix.
        body = re.sub(
            r"^[A-Za-z]{3}\s+\d{4}\s*[-–]?\s*"
            r"(?:[A-Za-z]{3}\s+\d{4})?\s*",
            "",
            body
        ).strip()

        # Split PDF bullets and common evidence verbs.
        parts = re.split(
            r"\s*•\s*|"
            r"(?=(?:Analyzed|Applied|Compared|Developed|"
            r"Extracted|Designed|Generated|Validated|Translated)\b)",
            body,
            flags=re.IGNORECASE
        )

        for part in parts:
            part = part.strip(" -–")
            if not part:
                continue

            projects.append({
                "source": "Project",
                "title": title,
                "text": part,
                "skills": extract_skills_with_aliases(
                    part,
                    skill_aliases
                )
            })

    return projects


project_bullets = extract_projects_from_cv(
    cv_profile["projects"]
)

for bullet in project_bullets:
    bullet["skills"] = extract_skills_with_aliases(
        bullet["text"],
        skill_aliases
    )

all_evidence = experience_bullets + project_bullets

print("\nEvidence bullets:", len(all_evidence))


# ============================================================



Evidence bullets: 20


In [9]:
# 8. JOB DESCRIPTION
# ============================================================
# Replace this entire block with the JD you want to analyse.

job_description = """
Data Analyst

Required Skills:
SQL
Python
Excel
Statistics
Hypothesis Testing
Data Visualization
Exploratory Data Analysis

Preferred Skills:
Forecasting
Machine Learning

Education:
Bachelor's degree in Statistics, Mathematics, Economics, Operations Research,
Engineering, Computer Science, Data Science or Business Analytics.

Experience:
2 years

Responsibilities:
Build dashboards and reports to communicate business insights.
Analyze large datasets to identify trends and patterns.
Use SQL and Python to perform data analysis.
Work with business stakeholders to support decision-making.
"""




In [10]:
# 9. JD configuration
# Keep these aligned with the JD above.

jd_required_skills = [
    "SQL",
    "Python",
    "Excel",
    "Statistics",
    "Hypothesis Testing",
    "Data Visualization",
    "Exploratory Data Analysis"
]

jd_preferred_skills = [
    "Forecasting",
    "Machine Learning"
]

jd_education = {
    "degree_level": "Bachelor's",
    "accepted_fields": [
        "Statistics",
        "Mathematics",
        "Economics",
        "Operations Research",
        "Engineering",
        "Computer Science",
        "Data Science",
        "Business Analytics"
    ]
}

jd_experience_required = 2.0

jd_responsibilities = [
    "Build dashboards and reports to communicate business insights.",
    "Analyze large datasets to identify trends and patterns.",
    "Use SQL and Python to perform data analysis.",
    "Work with business stakeholders to support decision-making."
]




In [11]:
# 10. Education matching
education_text = cv_profile["education"].lower()

accepted_fields = [
    field.lower()
    for field in jd_education["accepted_fields"]
]

field_match = any(
    field in education_text
    for field in accepted_fields
)

degree_match = (
    "bsc" in education_text
    or "bachelor" in education_text
    or "msc" in education_text
    or "master" in education_text
)

if field_match and degree_match:
    education_match = 100.0
elif field_match or degree_match:
    education_match = 50.0
else:
    education_match = 0.0

print("Education match:", education_match)




Education match: 100.0


In [12]:
# 11. Responsibility evidence rules
responsibility_evidence_map = {
    "Build dashboards and reports to communicate business insights.": {
        "skills": [
            "Power BI",
            "Data Visualization",
            "Analytical Reporting"
        ],
        "critical": [
            "Power BI",
            "Data Visualization"
        ]
    },

    "Analyze large datasets to identify trends and patterns.": {
        "skills": [
            "Large Datasets",
            "Data Processing",
            "Data Analysis",
            "Statistics",
            "Statistical Testing",
            "Exploratory Data Analysis"
        ],
        "critical": [
            "Large Datasets",
            "Statistics",
            "Statistical Testing",
            "Exploratory Data Analysis"
        ]
    },

    "Use SQL and Python to perform data analysis.": {
        "skills": [
            "SQL",
            "Python",
            "Data Analysis",
            "Data Processing"
        ],
        "critical": [
            "SQL",
            "Python"
        ]
    },

    "Work with business stakeholders to support decision-making.": {
        "skills": [
            "Decision Support",
            "Analytical Reporting"
        ],
        "critical": [
            "Decision Support"
        ]
    }
}


def classify_evidence(score):
    if score >= 80:
        return "Strong"
    elif score >= 50:
        return "Moderate"
    elif score > 0:
        return "Partial"
    return "No Evidence"




In [13]:
# 12. Direct text evidence
direct_evidence_aliases = {
    "Power BI": ["power bi", "powerbi"],
    "Data Visualization": [
        "data visualization",
        "visualization",
        "dashboard"
    ],
    "Large Datasets": [
        "large dataset",
        "large datasets",
        "7,043",
        "13,600",
        "155 countries",
        "195-variable"
    ],
    "Statistical Analysis": [
        "statistical analysis",
        "statistical techniques",
        "statistical findings",
        "inferential"
    ],
    "Exploratory Data Analysis": [
        "exploratory data analysis",
        "exploratory analysis"
    ],
    "SQL": ["sql"],
    "Python": ["python"],
    "Decision Support": [
        "decision support",
        "decision making",
        "decision-making",
        "client review",
        "actionable recommendations"
    ],
    "Analytical Reporting": [
        "analytical report",
        "analytical reports",
        "analytical outputs",
        "reporting"
    ]
}


def direct_evidence(bullet_text, critical_skills):
    text = bullet_text.lower()
    matches = []

    for skill in critical_skills:
        aliases = direct_evidence_aliases.get(
            skill,
            [skill]
        )

        if any(
            alias.lower() in text
            for alias in aliases
        ):
            matches.append(skill)

    return matches




In [14]:
# 13. Calculate responsibility evidence
relevant_results = []

for jd_item, requirement in responsibility_evidence_map.items():

    required = set(requirement["skills"])
    critical = set(requirement["critical"])

    best_score = -1
    best_bullet = None
    best_matches = []

    for bullet in all_evidence:

        bullet_skills = set(
            bullet.get("skills", [])
        )

        matched = bullet_skills.intersection(
            required
        )

        critical_matched = matched.intersection(
            critical
        )

        critical_score = (
            len(critical_matched)
            / len(critical)
            * 70
            if critical else 0
        )

        supporting = required - critical

        supporting_matched = matched.intersection(
            supporting
        )

        supporting_score = (
            len(supporting_matched)
            / len(supporting)
            * 30
            if supporting else 0
        )

        score = critical_score + supporting_score

        direct_matches = direct_evidence(
            bullet["text"],
            critical
        )

        if direct_matches:
            score = min(
                100,
                score + 10
            )

        if score > best_score:
            best_score = score
            best_bullet = bullet
            best_matches = list(matched)

    relevant_results.append({
        "jd": jd_item,
        "score": max(0, best_score),
        "evidence": best_bullet,
        "matched_skills": best_matches
    })


relevant_experience_score = (
    float(np.mean([
        x["score"]
        for x in relevant_results
    ]))
    if relevant_results
    else 0.0
)

print(
    "Relevant evidence score:",
    round(relevant_experience_score, 1)
)


# ============================================================


Relevant evidence score: 75.0


In [15]:
# 14. FINAL ANALYSIS FUNCTION
# ============================================================
def analyze_job_fit(cv, jd):

    cv_skills = set(
        cv.get("skills", [])
    )

    required_skills = set(
        jd["required_skills"]
    )

    preferred_skills = set(
        jd["preferred_skills"]
    )

    matched_required = (
        cv_skills.intersection(
            required_skills
        )
    )

    missing_required = (
        required_skills - cv_skills
    )

    matched_preferred = (
        cv_skills.intersection(
            preferred_skills
        )
    )

    missing_preferred = (
        preferred_skills - cv_skills
    )

    required_score = (
        len(matched_required)
        / len(required_skills)
        * 100
        if required_skills
        else 100
    )

    preferred_score = (
        len(matched_preferred)
        / len(preferred_skills)
        * 100
        if preferred_skills
        else 100
    )

    education_score = float(
        education_match
    )

    actual_experience = float(
        cv_experience_profile["total_years"]
    )

    required_experience = float(
        jd.get("experience", 0)
    )

    duration_score = (
        min(
            actual_experience
            / required_experience
            * 100,
            100
        )
        if required_experience > 0
        else 100
    )

    experience_score = (
        duration_score * 0.50
        + relevant_experience_score * 0.50
    )

    responsibility_results = []

    for item in relevant_results:

        evidence = item["evidence"]

        responsibility_results.append({
            "requirement": item["jd"],
            "score": round(
                float(item["score"]),
                1
            ),
            "strength": classify_evidence(
                item["score"]
            ),
            "source": (
                evidence["title"]
                if evidence
                else None
            ),
            "evidence": (
                evidence["text"]
                if evidence
                else None
            ),
            "matched_skills": item[
                "matched_skills"
            ]
        })

    responsibility_score = (
        float(np.mean([
            x["score"]
            for x in responsibility_results
        ]))
        if responsibility_results
        else 100
    )

    overall_score = (
        required_score * 0.40
        + preferred_score * 0.10
        + education_score * 0.10
        + experience_score * 0.15
        + responsibility_score * 0.25
    )

    if (
        required_score >= 80
        and education_score >= 50
    ):
        if actual_experience >= required_experience:
            eligibility = "Eligible"
        elif actual_experience >= required_experience * 0.75:
            eligibility = "Borderline"
        else:
            eligibility = "Experience Gap"

    elif (
        required_score >= 60
        and education_score >= 50
    ):
        eligibility = "Borderline"

    else:
        eligibility = "Not Eligible"

    if overall_score >= 85:
        competitiveness = "Strong Match"
    elif overall_score >= 70:
        competitiveness = "Good Match"
    elif overall_score >= 55:
        competitiveness = "Moderate Match"
    else:
        competitiveness = "Weak Match"

    strengths = []

    for skill in sorted(matched_required):
        strengths.append({
            "category": "Required Skill",
            "item": skill
        })

    for skill in sorted(matched_preferred):
        strengths.append({
            "category": "Preferred Skill",
            "item": skill
        })

    if education_score >= 100:
        strengths.append({
            "category": "Education",
            "item": "Education requirement satisfied"
        })

    gaps = []

    for skill in sorted(missing_required):
        gaps.append({
            "category": "Required Skill Gap",
            "item": skill
        })

    for skill in sorted(missing_preferred):
        gaps.append({
            "category": "Preferred Skill Gap",
            "item": skill
        })

    if actual_experience < required_experience:
        gaps.append({
            "category": "Experience Gap",
            "item": (
                f"JD requires {required_experience:.1f} years; "
                f"CV indicates {actual_experience:.1f} years."
            )
        })

    return {
        "role": jd["role"],
        "eligibility": eligibility,
        "competitiveness": competitiveness,
        "scores": {
            "required_skills": round(
                float(required_score),
                1
            ),
            "preferred_skills": round(
                float(preferred_score),
                1
            ),
            "education": round(
                float(education_score),
                1
            ),
            "experience": round(
                float(experience_score),
                1
            ),
            "responsibilities": round(
                float(responsibility_score),
                1
            ),
            "overall": round(
                float(overall_score),
                1
            )
        },
        "strengths": strengths,
        "gaps": gaps,
        "responsibility_analysis": responsibility_results
    }


# ============================================================


In [16]:
# 15. RUN FINAL ANALYSIS
# ============================================================
cv = {
    "skills": cv_skills,
    "education": cv_profile["education"],
    "experience": cv_profile["experience"],
    "projects": cv_profile["projects"]
}

jd = {
    "role": "Data Analyst",
    "required_skills": jd_required_skills,
    "preferred_skills": jd_preferred_skills,
    "education": jd_education,
    "experience": jd_experience_required,
    "responsibilities": jd_responsibilities
}

result = analyze_job_fit(
    cv,
    jd
)

print("\n" + "=" * 60)
print("AI JOB ANALYST — FINAL RESULT")
print("=" * 60)

print("\nEligibility:", result["eligibility"])
print("Competitiveness:", result["competitiveness"])

print("\nScores:")
for key, value in result["scores"].items():
    print(f"  {key}: {value}%")

print("\nStrengths:")
for item in result["strengths"]:
    print(
        f"  ✓ {item['category']}: "
        f"{item['item']}"
    )

print("\nGaps:")
for item in result["gaps"]:
    print(
        f"  • {item['category']}: "
        f"{item['item']}"
    )

print("\nResponsibility Analysis:")

for item in result["responsibility_analysis"]:

    print("\n", item["requirement"])
    print(
        "Score:",
        f"{item['score']}%",
        f"({item['strength']})"
    )
    print(
        "Source:",
        item["source"]
    )
    print(
        "Evidence:",
        item["evidence"]
    )



AI JOB ANALYST — FINAL RESULT

Eligibility: Borderline
Competitiveness: Strong Match

Scores:
  required_skills: 100.0%
  preferred_skills: 100.0%
  education: 100.0%
  experience: 75.0%
  responsibilities: 75.0%
  overall: 90.0%

Strengths:
  ✓ Required Skill: Data Visualization
  ✓ Required Skill: Excel
  ✓ Required Skill: Exploratory Data Analysis
  ✓ Required Skill: Hypothesis Testing
  ✓ Required Skill: Python
  ✓ Required Skill: SQL
  ✓ Required Skill: Statistics
  ✓ Preferred Skill: Forecasting
  ✓ Preferred Skill: Machine Learning
  ✓ Education: Education requirement satisfied

Gaps:
  • Experience Gap: JD requires 2.0 years; CV indicates 1.5 years.

Responsibility Analysis:

 Build dashboards and reports to communicate business insights.
Score: 80.0% (Strong)
Source: Customer Churn Prediction & Statistical Analysis
Evidence: developed a Power BI dashboard for customer-risk monitoring. Analysis of Alcohol Use Disorder (AUD) DataJan 2025 – Jun 2025

 Analyze large datasets to i

## Notes for GitHub

- Place a CV PDF beside this notebook and set `filename` in Section 1 before running.
- The example Job Description is defined in Section 8 and can be replaced with another JD.
- This notebook is the rule-based/basic version of the AI Job Analyst project.
